# 1.0 의도분류

> 실행 위치: 프로젝트 루트(`main.py`가 있는 폴더)에서 Jupyter를 실행합니다.

참고 노트북
- [0_3 Jev와 OpenAI 사용 방법](0_3_Jev와_LLM_사용_방법.ipynb)

고객은 **세션 ID·고객 정보·이번 입력**만 보냅니다. 전역 저장소에서 해당 세션의 마지막 상태를 복원해, 이전 의도를 참고하면서 이번 목적을 Jev로 다시 분류합니다.

- **입력:** `CustomerRequest(session_id, customer_info, input)`
- **처리:** 마지막 상태 복원 또는 새 상태 생성 → 이번 입력 반영 → Jev 분류 → 상태 저장
- **출력:** 이번 의도·신뢰도와 누적 기록이 담긴 `SessionState`

주문한 세션에서 “네, 맞아요”를 보내면 저장된 주문 의도를 참고합니다. 같은 세션에서 “근의공식 알려줘”를 보내면 새로 `unknown`을 선택해야 합니다.

```mermaid
flowchart LR
    input["Input<br/>CustomerRequest"]
    prepare["Python · 상태 복원/생성<br/>prepare_state(request)"]
    classify["매 턴 Jev 의도 분류<br/>classify_intent(state)"]
    save["Python · 상태 저장<br/>save_state(state)"]
    output["Output<br/>SessionState"]
    input --> prepare --> classify --> save --> output
    classDef io fill:#eeeeee,stroke:#999999,color:#333333;
    classDef python fill:#dbeafe,stroke:#3b82f6,color:#1e3a8a;
    class input,output io;
    class prepare,save python;
```

## 1. 스키마 설정

`schemas/session_state.py`의 공통 스키마를 불러옵니다.

고객 요청에는 `session_id`, `customer_info`, `input` 세 가지만 둡니다. 분류 결과와 대화 기록은 서버가 저장한 상태에서 이어받습니다.

| State 항목 | 역할 |
|---|---|
| `session_id` | 마지막 상태를 찾을 세션 ID |
| `customer` | 세션 상태에 저장할 고객 정보 |
| `input` | 이번 고객 입력 |
| `intent` | 이번 의도와 confidence |
| `previous_intent` | 마지막 상태에 있던 의도 이름 |
| `history` | 누적 대화 기록 |

In [1]:
# 1. 경로 준비: 프로젝트 루트에서 공통 스키마를 찾을 수 있도록 현재 폴더를 추가합니다.
import sys
import os
sys.path.append(os.getcwd())

# 2. 스키마 불러오기: 고객 요청과 세션 상태에 필요한 공통 자료형을 사용합니다.
from schemas.session_state import CustomerInput, CustomerRequest, HistoryMessage, IntentResult, SessionState

# 3. 스키마 확인: 공통 스키마 이름과 요청·상태에 들어갈 항목을 살펴봅니다.
print("불러온 스키마:", CustomerInput.__name__, CustomerRequest.__name__, HistoryMessage.__name__, IntentResult.__name__, SessionState.__name__)
print("고객 요청 항목:", list(CustomerRequest.model_fields))
print("세션 상태 항목:", list(SessionState.model_fields))


불러온 스키마: CustomerInput CustomerRequest HistoryMessage IntentResult SessionState
고객 요청 항목: ['session_id', 'customer_info', 'input']
세션 상태 항목: ['session_id', 'customer_id', 'customer', 'input', 'intent', 'previous_intent', 'history', 'search', 'recommendation', 'coupon', 'order', 'receipt', 'answer']


## 2. 모델 준비

`0_3`에서 사용한 Jev 클라이언트를 준비합니다.

In [2]:
# 1. 도구 준비: 모델 연결과 프롬프트 읽기, 실습 세션 구분에 필요한 도구를 불러옵니다.
from pathlib import Path
from uuid import uuid4

from typesafe_sdk import Choice, TypeSafeClient

# 2. 모델 연결: 환경변수의 키와 모델명으로 Jev 클라이언트를 준비합니다.
jev_model = os.getenv("TYPESAFE_MODEL", "jev-latest")
jev_client = TypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"], model=jev_model)
print("모델:", jev_model)

모델: jev-latest


## 3. 관련 프롬프트 불러오기

`prompts/intent_router.md`에서 **현재 문장 우선, 짧은 답변은 이전 의도 참고** 규칙을 읽습니다.

Jev에는 `text`, `previous_intent`, `has_image`만 보냅니다. 고객 정보·history·이전 confidence는 상태에만 보관합니다.

In [3]:
# 1. 지침 읽기: 분류 규칙 파일을 읽고 모델에 보낼 내용을 출력합니다.
intent_prompt = Path("prompts/intent_router.md").read_text(encoding="utf-8")
print(intent_prompt)

# 역할과 질문
당신은 모찌카페의 의도 분류기다. 이번 `text`의 목적을 판단해 `criteria`에서 의도 하나를 고른다. 이전 의도는 짧은 답변을 해석하는 참고값이며, 결과를 고정하는 값이 아니다.

## 입력
- `text`: 이번 고객 문장. 현재 목적을 판단할 가장 중요한 입력이다.
- `previous_intent`: 이전 의도 이름. 첫 대화이면 null이다.
- `has_image`: 이미지 첨부 여부. 원본 이미지는 제공되지 않는다.
- 서비스 입력에 `previous_intent` 대신 `session_state.intent.intent`가 있으면 그 값을 이전 의도로 참고한다.

## 판단 순서
1. 이번 문장에 분명한 새 목적이 있으면 이전 의도보다 이번 목적을 우선한다. 주문 중 "주차 방법을 알려주세요"는 product_consultation이다.
2. 이번 문장이 카페 기능과 무관하면 이전 의도가 있더라도 unknown이다. 주문 중 "근의공식 알려줘", "파이썬 코드를 짜줘", "내일 날씨 알려줘"는 unknown이다. 기존 작업에 억지로 연결하지 않는다.
3. "네", "네, 맞아요", "아니요", "두 잔이요", "아이스로요"처럼 자체 목적이 부족한 후속 답변은 유효한 이전 의도가 있으면 그 의도를 선택한다. 이전 product_order 뒤 "네, 맞아요"는 product_order다.
4. 이전 의도가 없거나 unknown인데 짧은 답변만 오면 unknown이다. 없는 주문이나 추천 대화를 가정하지 않는다.
5. 이전 의도가 product_recommendation이고 알레르기·취향을 말하면 추천의 후속 답변으로 본다. 이전 의도가 product_order이고 옵션·수량을 말하면 주문의 후속 답변으로 본다.
6. 상품과 수량을 말하며 주문하면 product_order다. 주문이 분명할 때 함께 말한 옵션이나 기피 재료 때문에 추천으로 바꾸지 않는다.
7. 상품의 이름·정체·후보를 찾는 요청은 product_search, 

In [4]:
# 1. 선택 기준: 서비스와 같은 여섯 가지 이름과 설명으로 Jev의 선택 범위를 정합니다.
criteria = {
    "product_search": "상품의 이름이나 정체를 알아내거나, 특정 상품을 찾아 후보를 확인하려는 목적이다.",
    "point_accumulation": "영수증의 결제 내역으로 포인트를 적립하거나 적립 내용을 확인한다.",
    "product_recommendation": "개인 조건에 맞는 상품을 고르거나 추천받으려는 요청 또는 진행 중인 추천의 알레르기·취향·선호에 대한 답변이다.",
    "product_consultation": "카페의 운영시간·위치·시설·이용 방법 등 카페 정보를 알아보려는 목적이다.",
    "product_order": "상품·수량·옵션을 주문하거나 진행 중인 주문 초안을 확인·수정한다.",
    "unknown": "어느 의도에도 맞지 않거나 맥락이 부족해 구분하기 어렵다. 여러 요청 중 먼저 처리할 것을 정할 수 없다."
}

# 2. 기준 확인: 분류할 의도 이름과 각 설명을 확인합니다.
for intent_name, description in criteria.items():
    print(f"{intent_name}: {description}")


product_search: 상품의 이름이나 정체를 알아내거나, 특정 상품을 찾아 후보를 확인하려는 목적이다.
point_accumulation: 영수증의 결제 내역으로 포인트를 적립하거나 적립 내용을 확인한다.
product_recommendation: 개인 조건에 맞는 상품을 고르거나 추천받으려는 요청 또는 진행 중인 추천의 알레르기·취향·선호에 대한 답변이다.
product_consultation: 카페의 운영시간·위치·시설·이용 방법 등 카페 정보를 알아보려는 목적이다.
product_order: 상품·수량·옵션을 주문하거나 진행 중인 주문 초안을 확인·수정한다.
unknown: 어느 의도에도 맞지 않거나 맥락이 부족해 구분하기 어렵다. 여러 요청 중 먼저 처리할 것을 정할 수 없다.


## 4. 사용자 input 준비

고객 요청을 직접 만듭니다. 같은 `session_id`로 다시 요청하면 저장된 마지막 상태를 읽습니다.

상태는 전역 딕셔너리 `state_store`에 `{세션 ID: [상태1, 상태2, ...]}`로 보관합니다. 아래 초기화 셀을 다시 실행하거나 커널을 재시작하면 비워집니다. 테스트용 세션 ID는 실행할 때마다 새로 만듭니다.

In [5]:
# 1. 저장소 준비: 이번 커널에서 세션별 상태 목록을 보관할 전역 딕셔너리를 만듭니다.
state_store = {}
test_run_id = uuid4().hex[:8]

# 2. 요청 준비: 고객은 이전 의도나 기록을 보내지 않고 이번 입력만 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_order",
    customer_info={"customer_id": "practice_customer", "name": "실습 고객"},
    input=CustomerInput(text="아이스 아메리카노 두 잔 주세요"),
)
print("고객 요청:")
print(request.model_dump_json(indent=2))

고객 요청:
{
  "session_id": "88db30ee_order",
  "customer_info": {
    "customer_id": "practice_customer",
    "name": "실습 고객",
    "birth_year": null,
    "membership_level": null,
    "allergies": null,
    "points": null,
    "stamps": null,
    "preferences": null
  },
  "input": {
    "text": "아이스 아메리카노 두 잔 주세요",
    "has_image": false
  }
}


## 5. 함수 만들기

### 상태 복원 또는 생성

세션의 마지막 상태가 있으면 복원하고, 마지막 `intent`를 `previous_intent`로 옮깁니다. 없으면 빈 의도·기록으로 시작합니다.

이번 입력으로 교체한 뒤 고객 문장을 history에 추가합니다. `customer_info=None`인 후속 요청은 기존 고객 정보를 유지합니다.

In [6]:
def prepare_state(request: CustomerRequest):
    """고객 요청으로 세션의 마지막 상태를 복원하거나 새로 만들고, 이번 입력과 기록을 채워 돌려줍니다."""
    # 1. 기록 조회: 전역 저장소에서 이번 세션에 저장된 상태 목록을 찾습니다.
    snapshots = state_store.get(request.session_id, [])

    # 2. 상태 준비: 마지막 결과를 복원하거나 첫 요청의 상태를 새로 만듭니다.
    if snapshots:
        state = SessionState.model_validate(snapshots[-1])

        ## 직전 분류 이름을 남기고 이번 입력으로 교체합니다.
        state.previous_intent = state.intent.intent if state.intent else None
        state.intent = None
        state.input = request.input

        ## 이번 요청에 고객 정보가 있을 때만 기존 정보를 갱신합니다.
        if request.customer_info is not None:
            state.customer = request.customer_info
    else:

        ## 저장 기록이 없는 세션은 이번 요청으로 시작합니다.
        state = SessionState(
            session_id=request.session_id,
            customer=request.customer_info,
            input=request.input,
        )

    # 3. 입력 기록: 이번 고객 문장을 누적 기록에 한 번 추가하고 상태를 돌려줍니다.
    state.history.append(HistoryMessage(role="user", content=request.input.text))

    return state

### Jev로 의도 분류

준비된 상태의 이번 입력과 이전 의도만 모델에 보내고, 반환된 의도와 신뢰도를 같은 상태에 채웁니다. 매 턴 Jev를 호출합니다.

In [7]:
def classify_intent(state: SessionState):
    """이번 입력과 저장소에서 가져온 이전 의도를 Jev에 보내, 새 분류 결과를 채운 같은 상태를 돌려줍니다."""
    # 1. 입력 준비: 고객 정보와 대화 기록을 제외하고 분류에 필요한 값만 묶습니다.
    user_input = {
        "text": state.input.text,
        "previous_intent": state.previous_intent,
        "has_image": state.input.has_image,
    }

    # 2. 질문 준비: 파일의 분류 지침과 여섯 가지 선택 기준을 묶습니다.
    questions = {"intent": Choice(instructions=intent_prompt, criteria=criteria)}

    # 3. 모델 호출: 이전 의도가 있어도 이번 문장의 목적을 Jev에 다시 묻습니다.
    response = jev_client.system_one(state=user_input, questions=questions)
    answer = response.answers["intent"]

    # 4. 결과 확인: 이번 의도와 신뢰도를 상태에 채우고 값이 있는지 확인합니다.
    state.intent = IntentResult(intent=answer.choice, confidence=answer.confidence)
    if state.intent.confidence is None:
        raise ValueError("Jev가 신뢰도를 반환하지 않았습니다.")

    return state

### 상태 저장

전역 `state_store`의 세션 목록 뒤에 이번 상태 스냅샷을 추가합니다. 파일은 사용하지 않습니다. 같은 상태를 실수로 두 번 저장하면 두 번째는 추가하지 않습니다.

이 단계에서는 의도만 분류하므로 실제 챗봇 답변을 만들어 history에 넣지는 않습니다. 이후 기능에서 생성한 답변도 같은 history에 담을 수 있습니다.

In [8]:
def save_state(state: SessionState):
    """처리한 상태를 전역 저장소의 세션 목록에 추가하고, 같은 스냅샷의 중복 저장을 막아 상태를 돌려줍니다."""
    # 1. 저장 준비: 전역 저장소에서 이번 세션의 목록을 찾고 독립된 상태 스냅샷을 만듭니다.
    snapshots = state_store.setdefault(state.session_id, [])
    snapshot = state.model_dump(mode="json")

    # 2. 중복 확인: 이미 저장한 바로 그 상태이면 다시 추가하지 않습니다.
    if snapshots and snapshots[-1] == snapshot:

        return state

    # 3. 상태 저장: 이번 스냅샷을 목록 뒤에 추가하고 같은 상태를 돌려줍니다.
    snapshots.append(snapshot)

    return state

## 6. 연결 테스트

각 셀에서 **고객 Input → 상태 준비·분류·저장 함수 → Output**을 확인합니다. 후속 요청에도 고객은 세션 ID와 이번 입력만 보냅니다.

### 테스트 1 - 새 주문 세션

저장된 상태가 없으므로 새 상태를 만들고 주문 의도를 저장합니다.

In [9]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_order",
    customer_info={"customer_id": "practice_customer", "name": "실습 고객"},
    input=CustomerInput(text="아이스 아메리카노 두 잔 주세요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("상태 준비 결과:", state.model_dump_json(indent=2))
state = classify_intent(state)
print("분류 결과:", state.intent.model_dump_json(indent=2))
state = save_state(state)
print("저장된 상태 수:", len(state_store[request.session_id]))

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_order
Input: 아이스 아메리카노 두 잔 주세요

Function: prepare_state → classify_intent → save_state
상태 준비 결과: {
  "session_id": "88db30ee_order",
  "customer_id": null,
  "customer": {
    "customer_id": "practice_customer",
    "name": "실습 고객",
    "birth_year": null,
    "membership_level": null,
    "allergies": null,
    "points": null,
    "stamps": null,
    "preferences": null
  },
  "input": {
    "text": "아이스 아메리카노 두 잔 주세요",
    "has_image": false
  },
  "intent": null,
  "previous_intent": null,
  "history": [
    {
      "role": "user",
      "content": "아이스 아메리카노 두 잔 주세요"
    }
  ],
  "search": null,
  "recommendation": null,
  "coupon": null,
  "order": null,
  "receipt": null,
  "answer": null
}


분류 결과: {
  "intent": "product_order",
  "confidence": 1.0
}
저장된 상태 수: 1

Output: product_order
Confidence: 1.0
History count: 1


### 테스트 2 - 새 상품검색 세션

다른 세션에는 별도의 상태를 만듭니다. 사진 내용은 보내지 않습니다.

In [10]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_search",
    input=CustomerInput(text="이 사진 속 메뉴 이름을 찾아주세요", has_image=True),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_search
Input: 이 사진 속 메뉴 이름을 찾아주세요

Function: prepare_state → classify_intent → save_state
Previous intent: None

Output: product_search
Confidence: 1.0
History count: 1


### 테스트 3 - 새 포인트 적립 세션

영수증 포인트 적립 요청을 분류합니다.


In [11]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_points",
    input=CustomerInput(text="이 영수증으로 포인트를 적립해 주세요", has_image=True),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_points
Input: 이 영수증으로 포인트를 적립해 주세요

Function: prepare_state → classify_intent → save_state
Previous intent: None



Output: point_accumulation
Confidence: 1.0
History count: 1


### 테스트 4 - 새 추천 세션

추천 대화를 시작하고 결과를 저장합니다.

In [12]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_recommend",
    customer_info={"customer_id": "practice_customer", "name": "실습 고객"},
    input=CustomerInput(text="달지 않은 음료를 추천해 주세요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_recommend
Input: 달지 않은 음료를 추천해 주세요

Function: prepare_state → classify_intent → save_state
Previous intent: None



Output: product_recommendation
Confidence: 1.0
History count: 1


### 테스트 5 - 새 카페 문의 세션

운영시간과 주차는 같은 카페 정보 문의입니다.

In [13]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_info",
    input=CustomerInput(text="주말 운영시간과 주차 방법을 알려주세요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_info
Input: 주말 운영시간과 주차 방법을 알려주세요

Function: prepare_state → classify_intent → save_state
Previous intent: None



Output: product_consultation
Confidence: 1.0
History count: 1


### 테스트 6 - 새 세션의 확인 답변

아직 이전 의도가 없는 세션에서 “네, 맞아요”는 unknown입니다.

In [14]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_no_context",
    input=CustomerInput(text="네, 맞아요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_no_context
Input: 네, 맞아요

Function: prepare_state → classify_intent → save_state
Previous intent: None



Output: unknown
Confidence: 1.0
History count: 1


### 테스트 7 - 저장된 추천 세션 이어가기

테스트 4와 같은 세션 ID로 요청합니다. 이전 의도와 고객 정보는 저장소에서 가져옵니다.

In [15]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_recommend",
    input=CustomerInput(text="우유 알레르기가 있어요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_recommend
Input: 우유 알레르기가 있어요

Function: prepare_state → classify_intent → save_state
Previous intent: product_recommendation



Output: product_recommendation
Confidence: 1.0
History count: 2


### 테스트 8 - 새 세션의 복합 요청

서로 다른 목적 중 먼저 할 일이 정해지지 않았으므로 unknown을 확인합니다.

In [16]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_mixed",
    input=CustomerInput(text="메뉴 추천해주고 주차도 알려주세요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_mixed
Input: 메뉴 추천해주고 주차도 알려주세요

Function: prepare_state → classify_intent → save_state
Previous intent: None



Output: unknown
Confidence: 1.0
History count: 1


### 테스트 9 - 저장된 주문 세션의 확인 답변

테스트 1의 주문 세션을 복원하고 “네, 맞아요”를 주문의 후속 답변으로 분류합니다.

In [17]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_order",
    input=CustomerInput(text="네, 맞아요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_order
Input: 네, 맞아요

Function: prepare_state → classify_intent → save_state
Previous intent: product_order



Output: product_order
Confidence: 1.0
History count: 2


### 테스트 10 - 주문 세션에서 수학 질문

같은 주문 세션이지만 이번 입력이 카페와 무관하므로 unknown으로 바뀌어야 합니다.

In [18]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_order",
    input=CustomerInput(text="근의공식 알려줘"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_order
Input: 근의공식 알려줘

Function: prepare_state → classify_intent → save_state
Previous intent: product_order



Output: unknown
Confidence: 1.0
History count: 3


### 테스트 11 - 같은 세션에서 새 카페 문의

이번에는 주차 문의를 새로 분류하고, 입력과 상태 기록을 계속 저장합니다.

In [19]:
# 1. Input 준비: 고객은 세션 ID와 이번 입력을 전달합니다.
request = CustomerRequest(
    session_id=f"{test_run_id}_order",
    input=CustomerInput(text="카페 주차 방법을 알려주세요"),
)
print("Session:", request.session_id)
print("Input:", request.input.text)

# 2. 함수 호출: 마지막 상태를 복원하거나 만들고, Jev 분류 후 이번 상태를 저장합니다.
print()
print("Function: prepare_state → classify_intent → save_state")
state = prepare_state(request)
print("Previous intent:", state.previous_intent)
state = classify_intent(state)
state = save_state(state)

# 3. Output 확인: 이번 의도·신뢰도와 누적 고객 입력 개수를 텍스트로 확인합니다.
print()
print("Output:", state.intent.intent)
print("Confidence:", state.intent.confidence)
print("History count:", len(state.history))

Session: 88db30ee_order
Input: 카페 주차 방법을 알려주세요

Function: prepare_state → classify_intent → save_state
Previous intent: unknown



Output: product_consultation
Confidence: 1.0
History count: 4
